In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf

spark = (
    SparkSession.builder
    .appName("KNN_PCA_K25")
    .master("local[*]")
    .config('spark.ui.port', '4050')
    .config("spark.driver.memory", "35g")
    .config("spark.driver.memoryOverhead", "4g")

    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")

    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

    .getOrCreate()
)

spark


In [3]:
parquet_path_raw = "/content/drive/MyDrive/bigdata/cic_ddos_raw.parquet"
parquet_path_25 = r"/content/drive/MyDrive/bigdata/cic_ddos_pca_25.parquet"
parquet_path_21 = r"/content/drive/MyDrive/bigdata/cic_ddos_pca_21.parquet"
parquet_path_27 = r"/content/drive/MyDrive/bigdata/cic_ddos_rf_27.parquet"

df_feature_raw = spark.read.parquet(parquet_path_raw).cache()
df_feature_25 = spark.read.parquet(parquet_path_25).cache()
df_feature_21 = spark.read.parquet(parquet_path_21).cache()
df_feature_27 = spark.read.parquet(parquet_path_27).cache()

df_feature_25.printSchema(), df_feature_21.printSchema(), df_feature_27.printSchema(), df_feature_raw.printSchema()

root
 |-- pca_k25: vector (nullable = true)
 |-- label: string (nullable = true)

root
 |-- pca_k21: vector (nullable = true)
 |-- label: string (nullable = true)

root
 |-- label: string (nullable = true)
 |-- rf_27: vector (nullable = true)

root
 |-- features: vector (nullable = true)
 |-- label: string (nullable = true)



(None, None, None, None)

In [4]:
from pyspark.ml.feature import StringIndexer
label_indexer = StringIndexer(
    inputCol="label",
    outputCol="label_index",
    handleInvalid="skip"
)
df_indexed_25 = label_indexer.fit(df_feature_25).transform(df_feature_25)
df_indexed_21 = label_indexer.fit(df_feature_21).transform(df_feature_21)
df_indexed_27 = label_indexer.fit(df_feature_27).transform(df_feature_27)
df_indexed_raw = label_indexer.fit(df_feature_raw).transform(df_feature_raw)


In [5]:
from pyspark.sql.functions import lit

def split_train_test(df_indexed):
  fractions = (
      df_indexed
      .select("label")
      .distinct()
      .withColumn("fraction", lit(0.02))
      .rdd.collectAsMap()
  )

  test_df = df_indexed.sampleBy(
      "label",
      fractions,
      seed=42
  )
  return test_df, df_indexed.subtract(test_df)

test_df_25, train_df_25 = split_train_test(df_indexed_25)
test_df_21, train_df_21 = split_train_test(df_indexed_21)
test_df_27, train_df_27 = split_train_test(df_indexed_27)
test_df_raw, train_df_raw = split_train_test(df_indexed_raw)

In [10]:
from pyspark.ml.clustering import KMeans

def kmeans_init(feature_name):
    return KMeans(
        featuresCol=feature_name,
        predictionCol="prediction",
        k=10,
        seed=42,
    )
kmeans_raw = kmeans_init("features")
kmeans_model_raw = kmeans_raw.fit(test_df_raw)

kmeans_k21 = kmeans_init("pca_k21")
kmeans_model_k21 = kmeans_k21.fit(test_df_21)

kmeans_k25 = kmeans_init("pca_k25")
kmeans_model_k25 = kmeans_k25.fit(test_df_25)

kmeans_k27 = kmeans_init("rf_27")
kmeans_model_k27 = kmeans_k27.fit(test_df_27)


In [11]:
kmeans_model_raw_path = r"/content/drive/MyDrive/bigdata/model/kmeans_cic_ddos_2019_raw"
kmeans_model_raw.write().overwrite().save(kmeans_model_raw_path)

kmeans_model_k21_path = r"/content/drive/MyDrive/bigdata/model/kmeans_cic_ddos_2019_pca_k21"
kmeans_model_k21.write().overwrite().save(kmeans_model_k21_path)

kmeans_model_k25_path = r"/content/drive/MyDrive/bigdata/model/kmeans_cic_ddos_2019_pca_k25"
kmeans_model_k25.write().overwrite().save(kmeans_model_k25_path)

kmeans_model_k27_path = r"/content/drive/MyDrive/bigdata/model/kmeans_cic_ddos_2019_rf_k27"
kmeans_model_k27.write().overwrite().save(kmeans_model_k27_path)
